In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cvxpy as cp
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, x_r, theta_0):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

In [4]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset, base_model: NN, X_train):
    alpha = params['alpha']
    lamb = params['lamb']
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    if recourse.name == "ROAR":
        print(weights_0, bias_0, theta_0)
    n = len(X)
    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]
        
        # LIME approximation of original NN
        np.random.seed(i)
        weights_0, bias_0 = lime_explanation(base_model.predict, X_train, x_0)
        weights_0, bias_0 = np.round(weights_0, 4), np.round(bias_0, 4)
        theta_0 = np.hstack((weights_0, bias_0))
        
        # Initalize recourse methods with theta_0
        recourse.set_weights(weights_0)
        recourse.set_bias(bias_0)
        
        x_r = recourse.get_recourse(x_0)
        append_result(results, recourse.name, seed, alpha, lamb, i, x_0, x_r, theta_0)

    df_results = pd.DataFrame(results)
    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f'../results/recourse/nn_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl')
    
    return df_results

In [5]:
def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    lamb = params['lamb']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = NN(X_train.shape[1])
        base_model.train(X_train.values, y_train.values)
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)

        # rng = np.random.default_rng(seed=seed)
        # size_N = int(np.rint(0.15 * recourse_needed_X_test.shape[0]))
        # recourse_needed_X_test = rng.choice(recourse_needed_X_test, size=size_N, replace=False) 
        
        for recourse_fn in recourse_fns:
            recourse = recourse_fn(weights=None, bias=None, alpha=alpha, lamb=lamb)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']
            
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset, base_model, X_train)
            results.append(df_results)

In [6]:
def run_experiment2(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    lamb = params['lamb']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = NN(X_train.shape[1])
        base_model.train(X_train.values, y_train.values)
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)

        rng = np.random.default_rng(seed=seed)
        size_N = int(np.rint(0.075 * recourse_needed_X_test.shape[0]))
        recourse_needed_X_test = rng.choice(recourse_needed_X_test, size=size_N, replace=False) 
        
        for recourse_fn in recourse_fns:
            recourse = recourse_fn(weights=None, bias=None, alpha=alpha, lamb=lamb)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']
            
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset, base_model, X_train)
            results.append(df_results)

In [10]:
alphas = [0.02, 0.1, 0.2, 0.3] # <------------------------
lambdas = [0.1, 0.7, 1.4, 2.1] # <------------------------

torch.manual_seed(0)

for lamb in lambdas:
    for alpha in alphas:
        
        d_results = {}
        params = {}
        params['alpha'] = alpha # float, None
        params['lamb'] = lamb
        params['seeds'] = range(5)
        params['save_results'] = True

        datasets = [SBADataset()] # <------------------------
        recourse_fns = [L1Recourse] # <------------------------

        for dataset in datasets:
            results = []
            print(f'Running {dataset.name} data...')
            run_experiment2(dataset, recourse_fns, params, results)
            
            d_results[dataset.name] = pd.concat(results)
            print(f'Finished {dataset.name}\n')

Running sba data...


[L1PSD] [alpha=0.02] [lambda=0.1]: 100%|██████████| 3/3 [10:44<00:00, 214.74s/it]


[L1PSD] Saving results for sba run 0


[L1PSD] [alpha=0.02] [lambda=0.1]: 100%|██████████| 3/3 [10:17<00:00, 205.99s/it]


[L1PSD] Saving results for sba run 1


...

[L1PSD] Saving results for sba run 2


[L1PSD] [alpha=0.3] [lambda=2.1]: 100%|██████████| 3/3 [30:54<00:00, 618.09s/it]


[L1PSD] Saving results for sba run 3


[L1PSD] [alpha=0.3] [lambda=2.1]: 100%|██████████| 3/3 [29:09<00:00, 583.03s/it]

[L1PSD] Saving results for sba run 4
Finished sba



In [ ]:
alphas = np.arange(0.02, 0.31 ,0.02).round(4) # <------------------------
lambdas = [0.1, 0.7, 1.4, 2.1] # <------------------------

torch.manual_seed(0)

for lamb in lambdas:
    for alpha in alphas:
        
        d_results = {}
        params = {}
        params['alpha'] = alpha # float, None
        params['lamb'] = lamb
        params['seeds'] = range(5)
        params['save_results'] = True

        datasets = [SyntheticDataset()] # <------------------------
        recourse_fns = [L1Recourse] # <------------------------

        for dataset in datasets:
            results = []
            print(f'Running {dataset.name} data...')
            run_experiment(dataset, recourse_fns, params, results)
            
            d_results[dataset.name] = pd.concat(results)
            print(f'Finished {dataset.name}\n')

Running synthetic data...


[L1PSD] [alpha=0.02] [lambda=0.1]: 100%|██████████| 96/96 [21:30<00:00, 13.45s/it]


[L1PSD] Saving results for synthetic run 0


[L1PSD] [alpha=0.02] [lambda=0.1]: 100%|██████████| 95/95 [27:27<00:00, 17.34s/it]


[L1PSD] Saving results for synthetic run 1


...

[L1PSD] Saving results for synthetic run 3


[L1PSD] [alpha=0.1] [lambda=0.1]: 100%|██████████| 105/105 [34:04<00:00, 19.47s/it]


[L1PSD] Saving results for synthetic run 4
Finished synthetic

Running synthetic data...


[L1PSD] [alpha=0.12] [lambda=0.1]:   8%|▊         | 8/96 [02:22<26:08, 17.83s/it]


KeyboardInterrupt: 